# Chapter 11: Point Pattern Analysis

*Part II — Geographic Data Science*

## Learning Objectives

By the end of this chapter you will be able to:

- Visualize a point pattern with scatter plots, hexbin density, and kernel density estimation (KDE)
- Summarize a point pattern's shape and spread with centrography: mean center, standard distance, and the standard deviational ellipse
- Test whether a point pattern is more clustered than pure chance would produce, using a quadrat test

## From Polygons to Points

Every chapter since Chapter 6 has worked with data that already lives inside a polygon — a municipality, a grid cell, a raster pixel. But plenty of real geographic data starts life as neither: a crime report, a disease case, a tree in a forest survey, a photograph's GPS tag. Each is a bare `(x, y)` location, with no polygon attached at all until an analyst decides to draw one.

Point pattern analysis studies that raw form directly, before any aggregation choice gets made. It asks two questions in particular: what does this scatter of points look like, summarized a few different ways — and is it more clustered (or more dispersed) than pure randomness would produce? Both questions have to be answered before choosing *how* to aggregate the points into the polygons the rest of Part II already knows how to analyze.

In [1]:
# Standard imports
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from pointpats import centrography, QStatistic

np.random.seed(1)

## A Synthetic Point Pattern

As in Chapters 11 through 13's synthetic grids, this chapter builds a point pattern with a *known* structure — two clusters — rather than starting from real coordinates. Knowing the ground truth is what lets the rest of the chapter confirm that each technique actually detects the structure that's really there, instead of taking a plausible-looking result on faith.

In [2]:
# Two clusters plus general scatter, in the same coordinate space
cluster_a = np.random.normal(loc=[2, 2], scale=0.5, size=(150, 2))
cluster_b = np.random.normal(loc=[7, 6], scale=0.8, size=(150, 2))
points = np.vstack([cluster_a, cluster_b])

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(points[:, 0], points[:, 1], s=10, alpha=0.6)
ax.set_title("Raw point pattern")
ax.set_aspect("equal")

## Visualizing Density: Hexbin and KDE

A plain scatter plot already shows two clusters here — but with a few thousand points instead of three hundred, individual markers overlap into an unreadable smear. Two techniques summarize density instead of plotting every point: `hexbin`, which counts points falling into hexagonal cells and colors by that count, and kernel density estimation (KDE), which smooths those counts into a continuous surface.

In [3]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

hb = axes[0].hexbin(points[:, 0], points[:, 1], gridsize=15, cmap="viridis")
axes[0].set_title("Hexbin density")
axes[0].set_aspect("equal")
plt.colorbar(hb, ax=axes[0])

kde = gaussian_kde(points.T)
xgrid, ygrid = np.mgrid[points[:, 0].min():points[:, 0].max():100j,
                        points[:, 1].min():points[:, 1].max():100j]
density = kde(np.vstack([xgrid.ravel(), ygrid.ravel()])).reshape(xgrid.shape)
axes[1].contourf(xgrid, ygrid, density, levels=20, cmap="viridis")
axes[1].set_title("KDE surface")
axes[1].set_aspect("equal")

plt.tight_layout()

`hexbin` is a direct count — easy to interpret, but its apparent smoothness depends entirely on `gridsize`, the same way a choropleth's story depends on its classification scheme (Chapter 15). KDE trades that dependency for a different one — a bandwidth parameter (`gaussian_kde`'s default, here) controlling how far each point's influence spreads — in exchange for a genuinely continuous surface rather than a grid of counts.

## Centrography: Summarizing a Pattern with a Few Numbers

Before testing whether a pattern is random, it helps to describe it: where is its center, how spread out is it, and does it stretch in one direction more than another? `pointpats.centrography` answers exactly these three questions.

In [4]:
mean_center = centrography.mean_center(points)
std_dist = centrography.std_distance(points)
semi_major, semi_minor, theta = centrography.ellipse(points)

print(f"Mean center: ({mean_center[0]:.2f}, {mean_center[1]:.2f})")
print(f"Standard distance: {std_dist:.2f}")
print(f"Ellipse semi-axes: {semi_major:.2f}, {semi_minor:.2f}  (theta={theta:.2f} rad)")

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(points[:, 0], points[:, 1], s=10, alpha=0.4)
ax.scatter(*mean_center, color="red", marker="x", s=150, label="Mean center")
circle = plt.Circle(mean_center, std_dist, fill=False, color="red", linestyle="--", label="Standard distance")
ax.add_patch(circle)
ax.set_aspect("equal")
ax.legend()
ax.set_title("Centrography summary")

The mean center lands between the two synthetic clusters — a reminder that a single summary point can be a poor description of a pattern that's actually bimodal. The standard-distance circle captures that same limitation visually: it has to be large enough to cover a fairly spread-out pattern, so a lot of genuinely empty space between the two clusters ends up inside it. Centrography is a fast, honest first look — not a substitute for actually visualizing the pattern first, which is exactly why it comes after the hexbin and KDE plots above, not before them.

## Is This Pattern Random? The Quadrat Test

Everything so far described the pattern that exists. This section asks a sharper question: if points had landed by pure chance — complete spatial randomness (CSR) — how likely is a pattern this clustered to occur? `QStatistic` answers this by dividing the study area into a grid of quadrats, counting points per quadrat, and comparing that count's variance to what CSR would predict via a chi-squared test.

To make the comparison meaningful, this section builds a second, genuinely random pattern in the same bounding box as the clustered one above — the fair test isn't clustered-versus-nothing, but clustered-versus-CSR in the identical study area.

In [5]:
random_points = np.random.uniform(
    low=points.min(axis=0), high=points.max(axis=0), size=(300, 2)
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(points[:, 0], points[:, 1], s=10, alpha=0.6)
axes[0].set_title("Clustered pattern")
axes[0].set_aspect("equal")
axes[1].scatter(random_points[:, 0], random_points[:, 1], s=10, alpha=0.6, color="orange")
axes[1].set_title("CSR (uniform random) pattern")
axes[1].set_aspect("equal")
plt.tight_layout()

In [6]:
for name, pts in [("Clustered", points), ("CSR", random_points)]:
    q = QStatistic(pts, shape="rectangle", nx=10, ny=10)
    print(f"{name:10s}  chi2 = {q.chi2:8.2f}   p-value = {q.chi2_pvalue:.4f}")

The clustered pattern's chi-squared statistic comes out in the thousands with a p-value indistinguishable from zero — CSR is rejected outright. The genuinely random pattern's p-value, by contrast, sits comfortably above any conventional significance threshold: exactly the outcome that confirms the test isn't simply flagging every pattern as clustered by default. The same quadrat grid used for this test, notice, is itself doing something close to the areal aggregation Chapter 12 assumes as its starting point — counting points per cell is one honest way to turn this chapter's raw locations into next chapter's per-unit variable.

## A Bridge to Chapter 12

Chapter 12 picks up exactly where the quadrat test's grid left off, but from the other direction: instead of asking whether points are randomly scattered across cells, it asks whether a variable *already aggregated* into areal units — one number per municipality or per grid cell — clusters in space more than chance would predict. Moran's I is the areal-data cousin of the quadrat test above; the counts this chapter's grid produced are exactly the kind of per-cell variable that test is built to examine.

## Exercises

1. **A single cluster.** Rebuild `points` using only `cluster_a` (drop `cluster_b` entirely) and rerun the KDE plot. Does the KDE surface still look meaningfully different from a CSR pattern covering the same bounding box?
2. **Quadrat resolution.** Rerun `QStatistic` on the clustered pattern with `nx=5, ny=5` instead of `nx=10, ny=10`. Does the test still reject CSR as strongly? What does that suggest about how sensitive the quadrat test is to the resolution chosen, compared to KDE's sensitivity to bandwidth?
3. **Centrography on each cluster separately.** Compute `centrography.mean_center` and `std_distance` for `cluster_a` and `cluster_b` individually, rather than the combined `points`. How much smaller is each cluster's standard distance compared to the combined pattern's?
4. **From points to a per-cell count.** Using the same 10×10 quadrat grid `QStatistic` builds internally, or a `numpy.histogram2d` on `points` with matching bins, produce a per-cell count array. This is the same kind of "one number per areal unit" data Chapter 12's ESDA techniques expect as input.

## Summary

### Key concepts introduced

- Point pattern analysis as the natural starting point for spatial data that hasn't been aggregated into polygons yet
- Density visualization with `hexbin` (discrete counts, sensitive to grid size) and KDE (continuous surface, sensitive to bandwidth)
- Centrography (`pointpats.centrography`): mean center, standard distance, and the standard deviational ellipse as fast, honest — but not sufficient on their own — summaries of a pattern's location and spread
- The quadrat test (`pointpats.QStatistic`) for testing complete spatial randomness (CSR), validated here against both a genuinely clustered and a genuinely random pattern in the same study area
- Quadrat counting as a bridge from raw point locations to the per-areal-unit variables Chapter 12's ESDA techniques are built to analyze

Chapter 12 returns to aggregated data and puts Chapter 10's spatial weights to their first real analytical use — measuring whether a variable already summarized per areal unit clusters in space more than chance alone would predict.

## Further Reading

- pointpats documentation, *Centrography*: <https://pysal.org/pointpats/notebooks/centrography.html>
- pointpats documentation, *Quadrat Statistics*: <https://pysal.org/pointpats/generated/pointpats.QStatistic.html>
- O'Sullivan, D., & Unwin, D. (2010). *Geographic Information Analysis* (2nd ed.). Wiley — Chapter 5 covers point pattern analysis and CSR testing in more depth than this chapter has room for